In [ ]:
# ===================================================================================
# WEB EXPERIENCE TEMPLATE MIGRATION (V22.3 - STRICT SHARING + GROUPS + IMAGES)
# - FIXES: Item Type issues by creating as "Web Experience Template" from the start.
# - FIXES: 404/Blank Screen by using the robust REST API upload logic.
# - FIXES: Solves "SharingGroupManager" error for groups.
# - UPDATED: Strict Sharing Mirroring (Private stays Private).
# - UPDATED: Mirrors Group Sharing (Matches Target Groups by Title).
# ===================================================================================

import json
import copy
import time
import csv
import os
import shutil
import datetime
import urllib3
import requests
import sys
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
# PUT YOUR TEMPLATE IDS HERE
# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_5_experience_templates.json")
if os.path.exists(_sidecar_path):
    import json as _json
    IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(IDS_TO_MIGRATE)} Template IDs from sidecar.")
else:
    IDS_TO_MIGRATE = [
        # Example Source ID
        # "4502ab95436640f8a8243d7e2b2fe3a9",
        # "f1607ea3cddd4318aeabfd348ff22a8e",
    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
# Initialize variables to None to prevent NameError
source_gis = None
target_gis = None

print("Connecting...")
try:
    session = requests.Session()
    # Robust retry strategy
    retry = Retry(
        total=5, 
        backoff_factor=2, 
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    
    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
    
    # Verify connections explicitly
    print(f"   > Source Connected: {source_gis.url}")
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f"   > Target Connected: {target_gis.users.me.username}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    print("   The script cannot continue because the GIS object failed to initialize.")
    sys.exit(1) # Force exit

# Double check safety before loop
if not source_gis or not target_gis:
    print("❌ GIS Initialization incomplete. Aborting.")
    sys.exit(1)

# =============================================================================
# --- LOGGING UTILS ------------------------------------------------------------
# =============================================================================
ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df = pd.read_csv(LOG_FILE)
        if 'SourceID' in df.columns: ALREADY_MIGRATED_IDS = set(df['SourceID'].astype(str).str.strip())
    except: pass
else:
    # Create log file if it doesn't exist
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def log_migration(source_id, target_id, title):
    try:
        with open(LOG_FILE, 'a', newline='') as f:
            csv.writer(f).writerow([source_id, "N/A", target_id, title, datetime.datetime.now(), "Web Experience Template (V22.3)"])
    except: pass

# =============================================================================
# --- HELPER: FOLDER & OWNER UTILS ---------------------------------------------
# =============================================================================
def get_source_folder_name(source_item):
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                if f['id'] == source_item.ownerFolder: return f['title']
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    """Ensure a folder exists for a specific user in Target."""
    try:
        user = target_gis.users.get(username)
        if not user: return False
        
        # Check existing folders (Handles Folder Objects AND Dicts)
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'):
                existing_folders.append(f.title)
            elif isinstance(f, dict):
                existing_folders.append(f.get('title'))
        
        if folder_name in existing_folders: return True
        
        # Create if missing
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except: return False

def assign_correct_owner_and_folder(source_item, target_item):
    """
    Tries to match Source Owner in Target.
    If found -> Moves to same folder name.
    If NOT found -> Moves to DEFAULT_OWNER (portaladm) and DEFAULT_FOLDER (migrate_test).
    """
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = DEFAULT_FOLDER

        # Check if Source Owner exists in Target
        found_user = target_gis.users.get(source_owner)
        
        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner
            
            # Try to get source folder name
            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None # Root
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
            # Ensure default folder exists for default user
            ensure_target_folder_exists(DEFAULT_OWNER, DEFAULT_FOLDER)

        # Reassign
        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

    except Exception as e:
        print(f"      ⚠ Reassign/Move Failed: {e} (Item remains with Creator)")

# =============================================================================
# --- HELPER: THUMBNAIL & SHARING ----------------------------------------------
# =============================================================================
def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbs"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: target_item.update(thumbnail=thumb_path)
    except: pass

def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. STRICTLY applies the same level to Target (No forcing Org).
    3. Maps Source Groups -> Target Groups (by exact Title).
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        
        # 1. Access Level
        is_public = False
        is_org = False
        
        if source_item.access == 'public':
            is_public = True
            is_org = True 
        elif source_item.access == 'org':
            is_org = True
        # Else: Private
            
        # 2. Map Groups (Robust retrieval)
        source_groups = []
        try:
            # Try getting via 'sharing.groups'
            raw_groups = source_item.sharing.groups
            # Explicitly check if it's a list. If not, fail to except block.
            if isinstance(raw_groups, list):
                source_groups = raw_groups
            else:
                raise ValueError("Not a list")
        except:
            # Fallback to 'shared_with'
            raw_shared = source_item.shared_with
            if isinstance(raw_shared, dict) and 'groups' in raw_shared:
                source_groups = raw_shared['groups']
            elif isinstance(raw_shared, list):
                source_groups = raw_shared

        target_group_ids = []
        
        if source_groups:
            print(f"         - Found {len(source_groups)} source groups. Mapping...")
            for sg in source_groups:
                # Handle Group Object vs String Name
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                # Search for group with exact title in Target
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")
        
        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- REST API HELPERS ---------------------------------------------------------
# =============================================================================
def remove_resource_via_rest(item_id, resource_path):
    url = f"{TARGET_URL}/sharing/rest/content/users/{target_gis.users.me.username}/items/{item_id}/removeResources"
    data = {"f": "json", "token": TARGET_TOKEN, "resource": resource_path}
    try: session.post(url, data=data, verify=False)
    except: pass

def add_resource_via_rest(item_id, local_file, file_name, prefix=None, overwrite=True):
    if overwrite and prefix:
        remove_resource_via_rest(item_id, f"{prefix}/{file_name}")

    url = f"{TARGET_URL}/sharing/rest/content/users/{target_gis.users.me.username}/items/{item_id}/addResources"
    with open(local_file, "rb") as f:
        data = {"f": "json", "token": TARGET_TOKEN, "fileName": file_name, "access": "inherit"}
        if prefix: data["resourcesPrefix"] = prefix 
        r = session.post(url, data=data, files={"file": f}, verify=False)
    
    if r.status_code != 200: raise Exception(f"HTTP {r.status_code}: {r.text}")
    return r.json()

def get_best_data(item):
    print("   🔍 Hunting for Draft Config...")
    try:
        url = f"{source_gis.url}/sharing/rest/content/items/{item.id}/resources/config/config.json"
        res = requests.get(url, params={'token': SOURCE_TOKEN}, verify=False)
        if res.status_code == 200 and res.json():
            print("      ✅ FOUND DRAFT CONFIG!")
            return res.json()
    except: pass
    print("      ℹ️ Draft fetch failed. Using Published Data.")
    return item.get_data()

def find_and_swap_ids(obj):
    if isinstance(obj, dict):
        if 'itemId' in obj:
            old_id = obj['itemId']
            try:
                res = target_gis.content.search(f'tags:"SourceID:{old_id}"', max_items=1)
                if res:
                    obj['itemId'] = res[0].id
                    if old_id == PROBLEM_SOURCE_ID:
                         if 'layerId' in obj and isinstance(obj['layerId'], int) and obj['layerId'] > 17:
                             obj['layerId'] -= 1
            except: pass
        for v in obj.values(): find_and_swap_ids(v)
    elif isinstance(obj, list):
        for item in obj: find_and_swap_ids(item)

def copy_resources_robust(src_item, tgt_item):
    print("   📂 Copying Assets (REST Mode)...")
    temp_dir = f"temp_res_{src_item.id}"
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)
    
    try:
        url = f"{source_gis.url}/sharing/rest/content/items/{src_item.id}/resources"
        res = requests.get(url, params={'f': 'json', 'token': SOURCE_TOKEN}, verify=False)
        resources = res.json().get('resources', [])

        count = 0
        for r_obj in resources:
            r_path = r_obj['resource'] if isinstance(r_obj, dict) else r_obj
            if "config/config.json" in r_path or r_path == "config.json": continue 
            
            # Download
            dl_url = f"{source_gis.url}/sharing/rest/content/items/{src_item.id}/resources/{r_path}"
            r_res = requests.get(dl_url, params={'token': SOURCE_TOKEN}, stream=True, verify=False)
            
            if r_res.status_code == 200:
                # Save
                clean_path = r_path.replace("\\", "/")
                local_path = os.path.join(temp_dir, clean_path.replace("/", os.sep))
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                with open(local_path, 'wb') as f:
                    for chunk in r_res.iter_content(chunk_size=8192): f.write(chunk)
                
                # Upload via REST to preserve folder (prefix)
                folder_part, file_part = os.path.split(clean_path)
                prefix = folder_part if folder_part else None
                
                try:
                    add_resource_via_rest(tgt_item.id, local_path, file_part, prefix=prefix, overwrite=True)
                    count += 1
                except: pass

        print(f"      ✅ Copied {count} assets.")
    except Exception as e:
        print(f"      ⚠ Resource Copy Warning: {e}")
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"Starting Migration Factory (V22.3 - Owner Map + Naming Flag + Strict Share)...")
start_time = time.time()
STATS = {"scanned": 0, "created": 0, "failed": 0, "skipped": 0}

for src_id in IDS_TO_MIGRATE:
    try:
        STATS["scanned"] += 1
        
        if str(src_id) in ALREADY_MIGRATED_IDS:
             print(f"\n[Skip] {src_id} already in log.")
             STATS["skipped"] += 1
             continue

        src_item = source_gis.content.get(src_id)
        if not src_item:
            print(f"❌ Source {src_id} not found.")
            STATS["failed"] += 1
            continue

        print(f"\nProcessing: {src_item.title}...")
        
        # 1. GET DATA
        data = get_best_data(src_item)
        if not data:
            print("   🛑 No data found.")
            STATS["failed"] += 1
            continue

        # --- DETERMINE TARGET TITLE ---
        target_title = f"{src_item.title} (Migrated Template)"
        if APPEND_MIGRATED:
            target_title = f"{src_item.title} (Migrated Template)" 

        # 2. CREATE TARGET ITEM (DIRECT TEMPLATE)
        print("   🏗️ Creating Web Experience Template...")
        final_tags = list(src_item.tags) if src_item.tags else []
        if "Migrated" not in final_tags: final_tags.append("Migrated")
        if f"SourceID:{src_id}" not in final_tags: final_tags.append(f"SourceID:{src_id}")

        # TEMPLATE KEYWORDS
        type_keywords = ["Experience Builder", "Template", "Web Experience Template"]

        props = {
            "type": "Web Experience Template", # <--- CREATING AS TEMPLATE DIRECTLY
            "title": target_title,
            "snippet": src_item.snippet,
            "description": src_item.description,
            "tags": final_tags,
            "typeKeywords": type_keywords,
            "text": "{}" # Temporary
        }

        new_app = None
        try:
            # Create in default folder initially (USE CONTENT.ADD)
            new_app = target_gis.content.add(props, folder=DEFAULT_FOLDER)
        except Exception as e:
            if "already exists" in str(e).lower():
                props["title"] = f"{target_title} {int(time.time())}"
                new_app = target_gis.content.add(props, folder=DEFAULT_FOLDER)
            else: raise e

        if new_app:
            # 3. PREPARE DATA (isTemplate = TRUE)
            print("   💉 Configuring as Template...")
            new_data = copy.deepcopy(data)
            find_and_swap_ids(new_data)
            
            # SET FLAG TRUE
            new_data["isTemplate"] = True # <--- CRITICAL FOR TEMPLATES
            
            json_str = json.dumps(new_data)
            patched_str = json_str.replace(src_id, new_app.id) 
            clean_src_url = SOURCE_URL.rstrip("/")
            clean_tgt_url = TARGET_URL.rstrip("/")
            if clean_src_url in patched_str:
                patched_str = patched_str.replace(clean_src_url, clean_tgt_url)

            # 4. UPDATE ITEM TEXT (Templates store data here too)
            new_app.update(item_properties={"text": patched_str, "url": ""})

            # 5. UPLOAD CONFIG VIA REST (The Fix for 404)
            try:
                temp_conf = f"temp_conf_{new_app.id}"
                os.makedirs(temp_conf, exist_ok=True)
                config_path = os.path.join(temp_conf, "config.json")
                with open(config_path, "w", encoding="utf-8") as f: f.write(patched_str)
                
                print("      📤 Uploading config via REST...")
                add_resource_via_rest(new_app.id, config_path, "config.json", prefix="config", overwrite=True)
            finally:
                shutil.rmtree(temp_conf, ignore_errors=True)

            # Copies images and other assets (ensures images are included)
            copy_resources_robust(src_item, new_app)
            
            # Copy Thumbnails
            copy_thumbnail(src_item, new_app)
            
            # Mirror Sharing (Strict Source)
            mirror_source_sharing(src_item, new_app)

            # 7. Owner & Folder Assignment
            assign_correct_owner_and_folder(src_item, new_app)

            log_migration(src_id, new_app.id, new_app.title)
            STATS["created"] += 1
            print(f"🚀 SUCCESS: Created Template '{new_app.title}'")

    except Exception as e:
        print(f"❌ Failed: {e}")
        STATS["failed"] += 1

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "="*60)
print("      TEMPLATE MIGRATION REPORT (V22.3)      ")
print("="*60)
print(f" ⏱️  Total Duration:            {duration_str}")
print("-" * 60)
print(f" 📝 Templates Scanned:        {STATS['scanned']}")
print(f" ⏭️  Skipped (Log):             {STATS['skipped']}")
print(f" ✅ Templates Created:        {STATS['created']}")
print(f" ❌ Failures:                 {STATS['failed']}")
print("="*60)